In [1]:
# ============================================
# 2일차 핵심 실습
# 1일차 결과물 불러오기 -> 여러 소스 읽기 -> merge -> 저장
# ============================================

In [2]:
import pandas as pd
import json
from pathlib import Path

In [3]:
# -------------------------------------------------
# 1. 폴더 설정
# -------------------------------------------------
base_dir = Path(".")
raw_dir = base_dir / "data" / "raw"
interim_dir = base_dir / "data" / "interim"

In [4]:
raw_dir.mkdir(parents=True, exist_ok=True)
interim_dir.mkdir(parents=True, exist_ok=True)

In [5]:
# -------------------------------------------------
# 2. 1일차 결과물 불러오기
# -------------------------------------------------
main_file = interim_dir / "ai4i_cleaned.csv"
df_main = pd.read_csv(main_file)

print("=" * 60)
print("1일차 결과물 불러오기")
print("=" * 60)
print("데이터 크기:", df_main.shape)
display(df_main.head())

1일차 결과물 불러오기
데이터 크기: (10000, 18)


,udi,product_id,type,air_temp_k,process_temp_k,rot_speed_rpm,torque_nm,tool_wear_min,machine_failure,twf,hdf,pwf,osf,rnf,temp_diff_k,power_index,tool_wear_level,tool_wear_outlier_flag
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0,10.5,66382.8,low,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0,10.5,65190.4,low,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0,10.4,74001.2,low,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0,10.4,56603.5,low,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0,10.5,56320.0,low,0


In [6]:
# -------------------------------------------------
# 3. 기준정보 CSV 만들기
# -------------------------------------------------
equipment_master = pd.DataFrame({
    "type": ["L", "M", "H"],
    "type_desc": ["low grade", "medium grade", "high grade"],
    "maintenance_cycle_days": [30, 20, 10],
    "inspection_level": ["basic", "standard", "strict"]
})

equipment_file = raw_dir / "equipment_master.csv"
equipment_master.to_csv(equipment_file, index=False, encoding="utf-8-sig")

In [7]:
# -------------------------------------------------
# 4. 기준정보 CSV 읽기
# -------------------------------------------------
df_equipment = pd.read_csv(equipment_file)

print("\n기준정보 데이터")
display(df_equipment)


기준정보 데이터


,type,type_desc,maintenance_cycle_days,inspection_level
0,L,low grade,30,basic
1,M,medium grade,20,standard
2,H,high grade,10,strict


In [8]:
# -------------------------------------------------
# 5. 메인 데이터 + 기준정보 결합
# -------------------------------------------------
df_merged = pd.merge(df_main, df_equipment, on="type", how="left")

print("\n기준정보 결합 후 데이터 크기:", df_merged.shape)
display(df_merged.head())

print("\n결합 후 결측치 확인")
display(df_merged[["type_desc", "maintenance_cycle_days", "inspection_level"]].isnull().sum())


기준정보 결합 후 데이터 크기: (10000, 21)


,udi,product_id,type,air_temp_k,process_temp_k,rot_speed_rpm,torque_nm,tool_wear_min,machine_failure,twf,...,pwf,osf,rnf,temp_diff_k,power_index,tool_wear_level,tool_wear_outlier_flag,type_desc,maintenance_cycle_days,inspection_level
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,...,0,0,0,10.5,66382.8,low,0,medium grade,20,standard
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,...,0,0,0,10.5,65190.4,low,0,low grade,30,basic
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,...,0,0,0,10.4,74001.2,low,0,low grade,30,basic
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,...,0,0,0,10.4,56603.5,low,0,low grade,30,basic
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,...,0,0,0,10.5,56320.0,low,0,low grade,30,basic



결합 후 결측치 확인


type_desc                 0
maintenance_cycle_days    0
inspection_level          0
dtype: int64

In [9]:
# -------------------------------------------------
# 6. 여러 CSV 파일 만들기
# -------------------------------------------------
daily_dir = raw_dir / "daily_targets"
daily_dir.mkdir(parents=True, exist_ok=True)

pd.DataFrame({
    "type": ["L", "M", "H"],
    "shift": ["A", "A", "A"],
    "daily_target": [120, 150, 180]
}).to_csv(daily_dir / "target_a.csv", index=False, encoding="utf-8-sig")

pd.DataFrame({
    "type": ["L", "M", "H"],
    "shift": ["B", "B", "B"],
    "daily_target": [110, 145, 175]
}).to_csv(daily_dir / "target_b.csv", index=False, encoding="utf-8-sig")

pd.DataFrame({
    "type": ["L", "M", "H"],
    "shift": ["C", "C", "C"],
    "daily_target": [100, 140, 170]
}).to_csv(daily_dir / "target_c.csv", index=False, encoding="utf-8-sig")

In [10]:
# -------------------------------------------------
# 7. 폴더 안 여러 CSV 읽어서 하나로 합치기
# -------------------------------------------------
file_list = list(daily_dir.glob("*.csv"))

df_list = []
for file in file_list:
    temp = pd.read_csv(file)
    temp["source_file"] = file.name
    df_list.append(temp)

df_targets = pd.concat(df_list, ignore_index=True)

print("\n여러 CSV를 합친 결과")
print("데이터 크기:", df_targets.shape)
display(df_targets)


여러 CSV를 합친 결과
데이터 크기: (9, 4)


,type,shift,daily_target,source_file
0,L,A,120,target_a.csv
1,M,A,150,target_a.csv
2,H,A,180,target_a.csv
3,L,C,100,target_c.csv
4,M,C,140,target_c.csv
5,H,C,170,target_c.csv
6,L,B,110,target_b.csv
7,M,B,145,target_b.csv
8,H,B,175,target_b.csv


In [11]:
# -------------------------------------------------
# 8. type별 평균 목표값 만들기
# -------------------------------------------------
df_target_summary = (
    df_targets
    .groupby("type", as_index=False)["daily_target"]
    .mean()
    .rename(columns={"daily_target": "avg_daily_target"})
)

print("\ntype별 평균 목표값")
display(df_target_summary)


type별 평균 목표값


,type,avg_daily_target
0,H,175.0
1,L,110.0
2,M,145.0


In [12]:
# -------------------------------------------------
# 9. 평균 목표값 결합
# -------------------------------------------------
df_merged = pd.merge(df_merged, df_target_summary, on="type", how="left")

print("\n평균 목표값 결합 후 데이터 크기:", df_merged.shape)
display(df_merged.head())


평균 목표값 결합 후 데이터 크기: (10000, 22)


,udi,product_id,type,air_temp_k,process_temp_k,rot_speed_rpm,torque_nm,tool_wear_min,machine_failure,twf,...,osf,rnf,temp_diff_k,power_index,tool_wear_level,tool_wear_outlier_flag,type_desc,maintenance_cycle_days,inspection_level,avg_daily_target
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,...,0,0,10.5,66382.8,low,0,medium grade,20,standard,145.0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,...,0,0,10.5,65190.4,low,0,low grade,30,basic,110.0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,...,0,0,10.4,74001.2,low,0,low grade,30,basic,110.0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,...,0,0,10.4,56603.5,low,0,low grade,30,basic,110.0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,...,0,0,10.5,56320.0,low,0,low grade,30,basic,110.0


In [13]:
# -------------------------------------------------
# 10. JSON 파일 만들기
# -------------------------------------------------
api_sample = [
    {"type": "L", "recommended_temp_range": "295~305K", "maintenance_priority": "low"},
    {"type": "M", "recommended_temp_range": "300~310K", "maintenance_priority": "medium"},
    {"type": "H", "recommended_temp_range": "305~315K", "maintenance_priority": "high"}
]

json_file = raw_dir / "api_sample.json"

with open(json_file, "w", encoding="utf-8") as f:
    json.dump(api_sample, f, ensure_ascii=False, indent=2)

In [14]:
# -------------------------------------------------
# 11. JSON 읽어서 DataFrame으로 변환
# -------------------------------------------------
with open(json_file, "r", encoding="utf-8") as f:
    api_data = json.load(f)

df_api = pd.DataFrame(api_data)

print("\nJSON 데이터")
display(df_api)


JSON 데이터


,type,recommended_temp_range,maintenance_priority
0,L,295~305K,low
1,M,300~310K,medium
2,H,305~315K,high


In [15]:
# -------------------------------------------------
# 12. JSON 데이터 결합
# -------------------------------------------------
df_merged = pd.merge(df_merged, df_api, on="type", how="left")

print("\nJSON 결합 후 데이터 크기:", df_merged.shape)
display(df_merged.head())


JSON 결합 후 데이터 크기: (10000, 24)


,udi,product_id,type,air_temp_k,process_temp_k,rot_speed_rpm,torque_nm,tool_wear_min,machine_failure,twf,...,temp_diff_k,power_index,tool_wear_level,tool_wear_outlier_flag,type_desc,maintenance_cycle_days,inspection_level,avg_daily_target,recommended_temp_range,maintenance_priority
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,...,10.5,66382.8,low,0,medium grade,20,standard,145.0,300~310K,medium
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,...,10.5,65190.4,low,0,low grade,30,basic,110.0,295~305K,low
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,...,10.4,74001.2,low,0,low grade,30,basic,110.0,295~305K,low
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,...,10.4,56603.5,low,0,low grade,30,basic,110.0,295~305K,low
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,...,10.5,56320.0,low,0,low grade,30,basic,110.0,295~305K,low


In [16]:
# -------------------------------------------------
# 13. 간단한 파생 변수 추가
# -------------------------------------------------
df_merged["high_wear_flag"] = (df_merged["tool_wear_min"] >= 150).astype(int)
df_merged["failure_risk_note"] = "normal"

df_merged.loc[
    (df_merged["machine_failure"] == 1) & (df_merged["high_wear_flag"] == 1),
    "failure_risk_note"
] = "check_now"

print("\n파생 변수 확인")
display(
    df_merged[["type", "machine_failure", "tool_wear_min", "high_wear_flag", "failure_risk_note"]].head(10)
)


파생 변수 확인


,type,machine_failure,tool_wear_min,high_wear_flag,failure_risk_note
0,M,0,0,0,normal
1,L,0,3,0,normal
2,L,0,5,0,normal
3,L,0,7,0,normal
4,L,0,9,0,normal
5,M,0,11,0,normal
6,L,0,14,0,normal
7,L,0,16,0,normal
8,M,0,18,0,normal
9,M,0,21,0,normal


In [17]:
# -------------------------------------------------
# 14. 결합 결과 요약
# -------------------------------------------------
print("\ntype별 평균 공구 마모")
display(
    df_merged.groupby("type", as_index=False)["tool_wear_min"]
    .mean()
    .round(2)
)

print("\nmaintenance_priority별 건수")
display(df_merged["maintenance_priority"].value_counts())


type별 평균 공구 마모


,type,tool_wear_min
0,H,107.42
1,L,108.38
2,M,107.27



maintenance_priority별 건수


maintenance_priority
low       6000
medium    2997
high      1003
Name: count, dtype: int64

In [18]:
# -------------------------------------------------
# 15. 최종 확인
# -------------------------------------------------
print("\n" + "=" * 60)
print("최종 데이터 확인")
print("=" * 60)
print("데이터 크기:", df_merged.shape)
print("최종 컬럼 목록")
print(df_merged.columns.tolist())

display(df_merged.head())


최종 데이터 확인
데이터 크기: (10000, 26)
최종 컬럼 목록
['udi', 'product_id', 'type', 'air_temp_k', 'process_temp_k', 'rot_speed_rpm', 'torque_nm', 'tool_wear_min', 'machine_failure', 'twf', 'hdf', 'pwf', 'osf', 'rnf', 'temp_diff_k', 'power_index', 'tool_wear_level', 'tool_wear_outlier_flag', 'type_desc', 'maintenance_cycle_days', 'inspection_level', 'avg_daily_target', 'recommended_temp_range', 'maintenance_priority', 'high_wear_flag', 'failure_risk_note']


,udi,product_id,type,air_temp_k,process_temp_k,rot_speed_rpm,torque_nm,tool_wear_min,machine_failure,twf,...,tool_wear_level,tool_wear_outlier_flag,type_desc,maintenance_cycle_days,inspection_level,avg_daily_target,recommended_temp_range,maintenance_priority,high_wear_flag,failure_risk_note
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,...,low,0,medium grade,20,standard,145.0,300~310K,medium,0,normal
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,...,low,0,low grade,30,basic,110.0,295~305K,low,0,normal
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,...,low,0,low grade,30,basic,110.0,295~305K,low,0,normal
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,...,low,0,low grade,30,basic,110.0,295~305K,low,0,normal
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,...,low,0,low grade,30,basic,110.0,295~305K,low,0,normal


In [19]:
# -------------------------------------------------
# 16. 저장
# -------------------------------------------------
output_file = interim_dir / "ai4i_enriched.csv"
df_merged.to_csv(output_file, index=False, encoding="utf-8-sig")

print("\n저장 완료:", output_file)


저장 완료: data/interim/ai4i_enriched.csv


In [20]:
# -------------------------------------------------
# 17. 저장 후 다시 확인
# -------------------------------------------------
check_df = pd.read_csv(output_file)

print("\n저장 후 다시 읽은 데이터 크기:", check_df.shape)
display(check_df.head())


저장 후 다시 읽은 데이터 크기: (10000, 26)


,udi,product_id,type,air_temp_k,process_temp_k,rot_speed_rpm,torque_nm,tool_wear_min,machine_failure,twf,...,tool_wear_level,tool_wear_outlier_flag,type_desc,maintenance_cycle_days,inspection_level,avg_daily_target,recommended_temp_range,maintenance_priority,high_wear_flag,failure_risk_note
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,...,low,0,medium grade,20,standard,145.0,300~310K,medium,0,normal
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,...,low,0,low grade,30,basic,110.0,295~305K,low,0,normal
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,...,low,0,low grade,30,basic,110.0,295~305K,low,0,normal
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,...,low,0,low grade,30,basic,110.0,295~305K,low,0,normal
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,...,low,0,low grade,30,basic,110.0,295~305K,low,0,normal


In [21]:
# -------------------------------------------------
# 18. 마무리 요약
# -------------------------------------------------
print("\n" + "=" * 60)
print("2일차 실습 요약")
print("=" * 60)
print("1) 1일차 정제 데이터를 다시 불러왔다.")
print("2) 기준정보 CSV를 읽고 결합했다.")
print("3) 여러 CSV 파일을 읽어 하나로 합쳤다.")
print("4) JSON 데이터를 읽어 결합했다.")
print("5) 통합 결과를 ai4i_enriched.csv 로 저장했다.")


2일차 실습 요약
1) 1일차 정제 데이터를 다시 불러왔다.
2) 기준정보 CSV를 읽고 결합했다.
3) 여러 CSV 파일을 읽어 하나로 합쳤다.
4) JSON 데이터를 읽어 결합했다.
5) 통합 결과를 ai4i_enriched.csv 로 저장했다.


In [22]:
# end